# Decision Trees and Ensemble Learning

## Credit Risk Scoring Project

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

## Data Cleaning & Preparation
    - Downloading the dataset
    - Reencodint the categorical variables
    - Doing test/train/split

In [2]:
data_url = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-06-trees/CreditScoring.csv'

In [3]:
df = pd.read_csv(data_url)
df.head()

,Status,Seniority,Home,Time,Age,Marital,Records,Job,Expenses,Income,Assets,Debt,Amount,Price
0,1,9,1,60,30,2,1,3,73,129,0,0,800,846
1,1,17,1,60,58,3,1,1,48,131,0,0,1000,1658
2,2,10,2,36,46,2,2,3,90,200,3000,0,2000,2985
3,1,0,1,60,24,1,1,1,63,182,2500,0,900,1325
4,1,0,1,36,26,1,1,1,46,107,0,0,310,910


In [4]:
df.columns

Index(['Status', 'Seniority', 'Home', 'Time', 'Age', 'Marital', 'Records',
       'Job', 'Expenses', 'Income', 'Assets', 'Debt', 'Amount', 'Price'],
      dtype='object')

In [5]:
df.columns = df.columns.str.lower()

In [6]:
df.columns

Index(['status', 'seniority', 'home', 'time', 'age', 'marital', 'records',
       'job', 'expenses', 'income', 'assets', 'debt', 'amount', 'price'],
      dtype='object')

In [7]:
df.isna().sum()

status       0
seniority    0
home         0
time         0
age          0
marital      0
records      0
job          0
expenses     0
income       0
assets       0
debt         0
amount       0
price        0
dtype: int64

In [8]:
df.status.value_counts()

1    3200
2    1254
0       1
Name: status, dtype: int64

In [9]:
status_values = {
    0: 'ok',
    1: 'default',
    2: 'unk'
}
df.status = df.status.map(status_values)

In [10]:
df.status.value_counts()

default    3200
unk        1254
ok            1
Name: status, dtype: int64

In [11]:
home_values = {
    1: 'rent',
    2: 'owner',
    3: 'private',
    4: 'ignore',
    5: 'parents',
    6: 'other',
    0: 'unk'
}

df.home = df.home.map(home_values)

marital_values = {
    1: 'single',
    2: 'married',
    3: 'widow',
    4: 'separated',
    5: 'divorced',
    0: 'unk'
}

df.marital = df.marital.map(marital_values)

records_values = {
    1: 'no',
    2: 'yes',
    0: 'unk'
}

df.records = df.records.map(records_values)

job_values = {
    1: 'fixed',
    2: 'partime',
    3: 'freelance',
    4: 'others',
    0: 'unk'
}

df.job = df.job.map(job_values)

In [12]:
df.head()

,status,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
0,default,9,rent,60,30,married,no,freelance,73,129,0,0,800,846
1,default,17,rent,60,58,widow,no,fixed,48,131,0,0,1000,1658
2,unk,10,owner,36,46,married,yes,freelance,90,200,3000,0,2000,2985
3,default,0,rent,60,24,single,no,fixed,63,182,2500,0,900,1325
4,default,0,rent,36,26,single,no,fixed,46,107,0,0,310,910


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4455 entries, 0 to 4454
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   status     4455 non-null   object
 1   seniority  4455 non-null   int64 
 2   home       4455 non-null   object
 3   time       4455 non-null   int64 
 4   age        4455 non-null   int64 
 5   marital    4455 non-null   object
 6   records    4455 non-null   object
 7   job        4455 non-null   object
 8   expenses   4455 non-null   int64 
 9   income     4455 non-null   int64 
 10  assets     4455 non-null   int64 
 11  debt       4455 non-null   int64 
 12  amount     4455 non-null   int64 
 13  price      4455 non-null   int64 
dtypes: int64(9), object(5)
memory usage: 487.4+ KB


In [14]:
df.describe().round()

,seniority,time,age,expenses,income,assets,debt,amount,price
count,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0,4455.0
mean,8.0,46.0,37.0,56.0,763317.0,1060341.0,404382.0,1039.0,1463.0
std,8.0,15.0,11.0,20.0,8703625.0,10217569.0,6344253.0,475.0,628.0
min,0.0,6.0,18.0,35.0,0.0,0.0,0.0,100.0,105.0
25%,2.0,36.0,28.0,35.0,80.0,0.0,0.0,700.0,1118.0
50%,5.0,48.0,36.0,51.0,120.0,3500.0,0.0,1000.0,1400.0
75%,12.0,60.0,45.0,72.0,166.0,6000.0,0.0,1300.0,1692.0
max,48.0,72.0,68.0,180.0,99999999.0,99999999.0,99999999.0,5000.0,11140.0


In [15]:
# replacing numerical columns max value with nan
for c in ['income', 'assets', 'debt']:
    print(c,df[c].max())

income 99999999
assets 99999999
debt 99999999


In [16]:
for c in ['income', 'assets', 'debt']:
    df[c] = df[c].replace(to_replace=99999999, value=np.nan)

In [17]:
for c in ['income', 'assets', 'debt']:
    print(c,df[c].max())

income 959.0
assets 300000.0
debt 30000.0


In [18]:
# dropping the unknown status rows from the dataset
df = df[df.status != 'unk'].reset_index(drop=True)

In [19]:
df.status.value_counts()

default    3200
ok            1
Name: status, dtype: int64

In [20]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [21]:
df_train.status.value_counts()

default    1919
ok            1
Name: status, dtype: int64

In [22]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [23]:
y_train = (df_train.status == 'default').astype('int').values
y_val = (df_val.status == 'default').astype('int').values
y_test = (df_test.status == 'default').astype('int').values

In [24]:
del df_train['status']
del df_test['status']
del df_val['status']

In [25]:
df_train

,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
0,8,owner,36,53,married,no,fixed,45,254.0,6000.0,0.0,350,575
1,3,owner,42,41,married,no,freelance,35,0.0,40000.0,0.0,1460,2191
2,2,rent,60,32,married,no,partime,87,115.0,2000.0,0.0,1250,1517
3,8,owner,36,37,married,yes,fixed,45,142.0,4500.0,300.0,370,870
4,15,owner,24,41,married,yes,partime,45,180.0,6000.0,0.0,700,1888
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1915,2,owner,60,45,married,no,freelance,75,75.0,2500.0,0.0,1100,1242
1916,0,parents,12,22,single,no,others,35,42.0,0.0,0.0,200,748
1917,14,owner,12,41,married,no,freelance,60,245.0,7000.0,0.0,500,1253
1918,20,other,48,47,married,no,fixed,45,182.0,0.0,0.0,1200,1431


## Decision Trees
    - How decision tree work?
    - training decision tree
    - overfitting
    - controlling the size of the tree

In [26]:
y_train

array([1, 1, 1, ..., 1, 1, 1])

In [27]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score

In [28]:
train_dicts = df_train.fillna(0).to_dict(orient='records')

In [29]:
train_dicts[:1]

[{'seniority': 8,
  'home': 'owner',
  'time': 36,
  'age': 53,
  'marital': 'married',
  'records': 'no',
  'job': 'fixed',
  'expenses': 45,
  'income': 254.0,
  'assets': 6000.0,
  'debt': 0.0,
  'amount': 350,
  'price': 575}]

In [30]:
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_train

array([[5.30e+01, 3.50e+02, 6.00e+03, ..., 0.00e+00, 8.00e+00, 3.60e+01],
       [4.10e+01, 1.46e+03, 4.00e+04, ..., 0.00e+00, 3.00e+00, 4.20e+01],
       [3.20e+01, 1.25e+03, 2.00e+03, ..., 0.00e+00, 2.00e+00, 6.00e+01],
       ...,
       [4.10e+01, 5.00e+02, 7.00e+03, ..., 0.00e+00, 1.40e+01, 1.20e+01],
       [4.70e+01, 1.20e+03, 0.00e+00, ..., 0.00e+00, 2.00e+01, 4.80e+01],
       [2.50e+01, 1.50e+03, 3.00e+03, ..., 0.00e+00, 1.00e+01, 4.20e+01]])

In [31]:
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [32]:
val_dicts = df_val.fillna(0).to_dict(orient='records')
X_val = dv.transform(val_dicts)
X_val

array([[3.00e+01, 7.00e+02, 7.00e+03, ..., 0.00e+00, 6.00e+00, 4.80e+01],
       [4.60e+01, 1.00e+03, 5.50e+03, ..., 0.00e+00, 1.00e+00, 6.00e+01],
       [4.00e+01, 1.25e+03, 0.00e+00, ..., 0.00e+00, 2.00e+00, 6.00e+01],
       ...,
       [2.50e+01, 1.30e+03, 2.50e+03, ..., 0.00e+00, 1.10e+01, 6.00e+01],
       [3.00e+01, 7.00e+02, 0.00e+00, ..., 0.00e+00, 2.00e+00, 3.60e+01],
       [3.80e+01, 3.00e+02, 7.00e+03, ..., 1.00e+00, 2.40e+01, 1.80e+01]])

In [33]:
y_pred = dt_model.predict_proba(X_val)[:, 0]
roc_auc_score(y_val, y_pred)

/opt/homebrew/Caskroom/miniforge/base/envs/mlenv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


nan

In [37]:
from sklearn.tree import export_text

In [36]:
print(export_text(dt_model, feature_names=dv.get_feature_names_out()))

|--- home=private <= 0.50
|   |--- class: 1
|--- home=private >  0.50
|   |--- age <= 25.50
|   |   |--- seniority <= 8.50
|   |   |   |--- class: 1
|   |   |--- seniority >  8.50
|   |   |   |--- class: 0
|   |--- age >  25.50
|   |   |--- class: 1



In [38]:
dt_model = DecisionTreeClassifier(max_depth=2)
dt_model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,2
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [39]:
y_pred = dt_model.predict_proba(X_val)[:, 0]
roc_auc_score(y_val, y_pred)

/opt/homebrew/Caskroom/miniforge/base/envs/mlenv/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


nan

In [40]:
print(export_text(dt_model, feature_names=dv.get_feature_names_out()))

|--- home=private <= 0.50
|   |--- class: 1
|--- home=private >  0.50
|   |--- marital=single <= 0.50
|   |   |--- class: 1
|   |--- marital=single >  0.50
|   |   |--- class: 1

